# Exercise: consume_01 — your first consumer

**Goal:** read the events you produced earlier and stream new ones live.

**Prerequisite:** run `exercise_01_produce_single.ipynb` first so there is data in `strom` and `wasser`.

## Step 1 — configure the consumer

A consumer needs three settings:
- `bootstrap.servers` — broker address
- `group.id` — consumer group; Kafka tracks the offset *per group*
- `auto.offset.reset` — where to start when no offset is stored yet

**Task:** pick a name for `group.id` (e.g. `'energy-monitor'`).

In [ ]:
from confluent_kafka import Consumer

conf = {
    'bootstrap.servers': 'redpanda:29092',
    'group.id':          None,             # TODO: choose a name
    'auto.offset.reset': 'earliest',       # start from the oldest stored event
}
consumer = Consumer(conf)
print('Consumer created.')

## Step 2 — subscribe

**Task:** subscribe to *both* `strom` and `wasser`.

```python
consumer.subscribe(['topic1', 'topic2'])
```

In [ ]:
# TODO: subscribe to 'strom' and 'wasser'

print('Subscribed.')

## Step 3 — read existing events (up to 20)

After running:
- What do `partition`, `offset`, `key`, `value` mean?
- Does the output match the events you sent?

In [ ]:
messages_read = 0
empty_polls   = 0

while messages_read < 20 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None:
        empty_polls += 1
        continue
    if msg.error():
        print(f'Error: {msg.error()}'); continue

    empty_polls    = 0
    messages_read += 1
    key   = msg.key().decode()   if msg.key()   else 'None'
    value = msg.value().decode() if msg.value() else 'None'
    print(f'[{messages_read}] {msg.topic()} P{msg.partition()} offset={msg.offset()}')
    print(f'     key={key}  value={value}')

print(f'Read {messages_read} message(s).')

## Task A — live streaming

This cell waits up to 30 seconds for new events.

1. Run it.
2. In another tab, run `exercise_01_produce_single.ipynb` and send a new event.
3. **Does it appear here immediately?**

In [ ]:
from datetime import datetime

print('Waiting for new events (30s)...')
empty_polls = 0
while empty_polls < 15:   # 15 × 2s = 30s
    msg = consumer.poll(2.0)
    if msg is None:
        empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0
    ts  = datetime.now().strftime('%H:%M:%S')
    key = msg.key().decode() if msg.key() else 'None'
    print(f'[{ts}] {msg.topic()} P{msg.partition()} offset={msg.offset()} key={key}')
    print(f'       {msg.value().decode()}')

print('No new messages.')

## Task B — change the group, read everything again

What happens if we use a *different* `group.id`? Will the new consumer re-read all events?

> Each consumer group has its own offsets. A brand-new group has none, so `earliest` kicks in.

In [ ]:
consumer.close()

# TODO: create a new Consumer with group.id='analytics' and auto.offset.reset='earliest'.
# Subscribe to ['strom', 'wasser'] and read up to 10 events.


In [ ]:
try:
    consumer.close()
except Exception:
    pass
print('Cleaned up.')